In [2]:
!pip install biotite

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 12.9 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 61.7 MB/s eta 0:00:00:00:01


In [ ]:
import os
import random
import itertools
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import biotite.structure.io.pdbx as pdbx
import biotite.database.rcsb as rcsb

# ==============================================================================
# CONSTANTS, CONFIGURATIONS & RESIDUE MAP
# ==============================================================================
RANDOM_SEED = 42
PROTEIN_RESIDUES = {
    "ALA", "ARG", "ASN", "ASP", "CYS", "GLN", "GLU", "GLY", "HIS", "ILE",
    "LEU", "LYS", "MET", "PHE", "PRO", "SER", "THR", "TRP", "TYR", "VAL"
}
NUCLEIC_RESIDUES = {"A", "C", "G", "U", "DA", "DC", "DG", "DT"}
ALL_RESIDUES = PROTEIN_RESIDUES | NUCLEIC_RESIDUES
RESIDUE_MAP = {res: idx + 1 for idx, res in enumerate(sorted(list(ALL_RESIDUES)))}  # Class 0: Background

DEFAULT_SAVE_DIR = "./pdb_data"
TRAIN_PDB_IDS = [
    "1ubq", "1a8o", "1bpi", "1cjg", "1eyy", "1hel", "1l2y", "1pga",
    "1shg", "1csp", "1a70", "1f9g", "2igd", "1ten", "1ycr", "3gbw"
]
TEST_PDB_ID = "1crn"
DEFAULT_BOX_SIZE = 8.0
GRID_SIZE = 32
DEFAULT_MAX_PEAKS = 128

# ==============================================================================
# UTILITIES & VECTORIZED RASTERIZATION (NON-CHUNKED)
# ==============================================================================

def download_pdb_cif(pdb_id: str) -> str:
    path = rcsb.fetch(pdb_id, "cif", DEFAULT_SAVE_DIR)
    return str(path[0]) if isinstance(path, list) else str(path)


def load_coords_biotite(filepath: str) -> tuple[torch.Tensor, torch.Tensor]:
    atoms = pdbx.get_structure(pdbx.CIFFile.read(filepath), model=1)
    valid = atoms[np.isin(atoms.res_name, list(ALL_RESIDUES))]
    coords = torch.tensor(valid.coord, dtype=torch.float32)
    res_indices = torch.tensor([RESIDUE_MAP[name] for name in valid.res_name], dtype=torch.long)
    return coords, res_indices


def coords_to_density(coords: torch.Tensor, sigma: float) -> torch.Tensor:
    if coords.shape[0] == 0:
        return torch.zeros((GRID_SIZE, GRID_SIZE, GRID_SIZE), device=coords.device)
    device = coords.device
    ticks = torch.linspace(0.0, DEFAULT_BOX_SIZE, GRID_SIZE, device=device)
    grid = torch.stack(torch.meshgrid(ticks, ticks, ticks, indexing='ij'), dim=-1).view(-1, 3)
    sq_dists = torch.sum((grid.unsqueeze(1) - coords.unsqueeze(0)) ** 2, dim=-1)
    density = torch.sum(torch.exp(-sq_dists / (2 * (sigma ** 2))), dim=-1)
    return density.view(GRID_SIZE, GRID_SIZE, GRID_SIZE)


def coords_to_binary_grid(coords: torch.Tensor) -> torch.Tensor:
    if coords.shape[0] == 0:
        return torch.zeros((GRID_SIZE, GRID_SIZE, GRID_SIZE), device=coords.device)
    device = coords.device
    ticks = torch.linspace(0.0, DEFAULT_BOX_SIZE, GRID_SIZE, device=device)
    grid = torch.stack(torch.meshgrid(ticks, ticks, ticks, indexing='ij'), dim=-1).view(-1, 3)
    dists = torch.norm(grid.unsqueeze(1) - coords.unsqueeze(0), dim=-1)
    return (dists.min(dim=-1)[0] <= 0.8).float().view(GRID_SIZE, GRID_SIZE, GRID_SIZE)


def coords_to_residue_grid(coords: torch.Tensor, res_indices: torch.Tensor) -> torch.Tensor:
    if coords.shape[0] == 0:
        return torch.zeros((GRID_SIZE, GRID_SIZE, GRID_SIZE), dtype=torch.long, device=coords.device)
    device = coords.device
    ticks = torch.linspace(0.0, DEFAULT_BOX_SIZE, GRID_SIZE, device=device)
    grid = torch.stack(torch.meshgrid(ticks, ticks, ticks, indexing='ij'), dim=-1).view(-1, 3)
    dists = torch.norm(grid.unsqueeze(1) - coords.unsqueeze(0), dim=-1)
    min_dists, nearest_indices = torch.min(dists, dim=-1)
    
    residue_grid = torch.zeros(grid.shape[0], dtype=torch.long, device=device)
    valid_mask = min_dists <= 0.8
    residue_grid[valid_mask] = res_indices[nearest_indices[valid_mask]]
    return residue_grid.view(GRID_SIZE, GRID_SIZE, GRID_SIZE)


def augment_batch_3d_joint(inputs: torch.Tensor, atom_targets: torch.Tensor, residue_targets: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    for dim in (-3, -2, -1):
        if random.random() > 0.5:
            inputs, atom_targets, residue_targets = torch.flip(inputs, [dim]), torch.flip(atom_targets, [dim]), torch.flip(residue_targets, [dim])
    for plane in [(-3, -2), (-2, -1), (-3, -1)]:
        if random.random() > 0.5:
            k = random.randint(1, 3)
            inputs = torch.rot90(inputs, k, dims=plane)
            atom_targets = torch.rot90(atom_targets, k, dims=plane)
            residue_targets = torch.rot90(residue_targets, k, dims=plane)
    return inputs, atom_targets, residue_targets


class BCEDiceLoss(nn.Module):
    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        bce = F.binary_cross_entropy(pred, target, reduction='mean')
        pred_flat, target_flat = pred.reshape(pred.shape[0], -1), target.reshape(target.shape[0], -1)
        intersection = torch.sum(pred_flat * target_flat, dim=-1)
        union = torch.sum(pred_flat, dim=-1) + torch.sum(target_flat, dim=-1)
        dice = 1.0 - (2.0 * intersection + 1e-6) / (union + 1e-6)
        return bce + dice.mean()


# ==============================================================================
# OPTIMIZED RESIDUAL 3D U-NET & PEAK FINDER
# ==============================================================================

class ConvBlock3d(nn.Module):
    def __init__(self, in_c: int, out_c: int) -> None:
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_c, out_c, 3, padding=1),
            nn.BatchNorm3d(out_c),
            nn.ReLU(True),
            nn.Conv3d(out_c, out_c, 3, padding=1),
            nn.BatchNorm3d(out_c)
        )
        self.shortcut = nn.Identity() if in_c == out_c else nn.Conv3d(in_c, out_c, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.relu(self.conv(x) + self.shortcut(x)) # Conv -> BN -> ReLU -> Conv -> BN + shortcut -> ReLU


class UNet3D(nn.Module):
    """Simplified Residual 3D U-Net preserving deep supervision and exact topology."""
    def __init__(self, in_channels: int = 1, out_channels: int = 1, init_features: int = 32) -> None:
        super().__init__()
        f = init_features
        # Encoder
        self.down1 = ConvBlock3d(in_channels, f)
        self.pool1 = nn.Conv3d(f, f, 3, stride=2, padding=1)
        self.down2 = ConvBlock3d(f, f * 2)
        self.pool2 = nn.Conv3d(f * 2, f * 2, 3, stride=2, padding=1)
        
        # Bottleneck
        self.bottleneck = ConvBlock3d(f * 2, f * 4)
        
        # Decoder
        self.up1 = nn.Upsample(scale_factor=2, mode="trilinear", align_corners=True)
        self.conv_up1 = ConvBlock3d(f * 6, f * 2)
        
        self.up2 = nn.Upsample(scale_factor=2, mode="trilinear", align_corners=True)
        self.conv_up2 = ConvBlock3d(f * 3, f)
        
        self.ds_conv1 = nn.Conv3d(f * 2, out_channels, 1)
        self.out_conv = nn.Conv3d(f, out_channels, 1)

    def forward(self, x: torch.Tensor, return_ds: bool = False) -> tuple[torch.Tensor, torch.Tensor] | torch.Tensor:
        x1 = self.down1(x)
        p1 = self.pool1(x1)
        x2 = self.down2(p1)
        p2 = self.pool2(x2)
        
        b = self.bottleneck(p2)
        
        u1 = self.up1(b)
        x3 = self.conv_up1(torch.cat([u1, x2], dim=1))
        
        u2 = self.up2(x3)
        x4 = self.conv_up2(torch.cat([u2, x1], dim=1))
        
        out = self.out_conv(x4)
        return (out, self.ds_conv1(x3)) if return_ds else out


class BatchedMeanShiftPeakFinder3D(nn.Module):
    """Simplified 3D peak finder utilizing local 3D Max-Pooling and greedy clash resolution."""
    def forward(self, density: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        B, _, X, Y, Z = density.shape
        device = density.device
        M = DEFAULT_MAX_PEAKS
        out_coords = torch.zeros((B, M, 3), device=device)
        out_values = torch.zeros((B, M), device=device)
        out_mask = torch.zeros((B, M), dtype=torch.bool, device=device)
        
        ticks = torch.linspace(0.0, DEFAULT_BOX_SIZE, X, device=device)
        grid = torch.stack(torch.meshgrid(ticks, ticks, ticks, indexing='ij'), dim=-1).view(-1, 3)
        
        max_pooled = F.max_pool3d(density, kernel_size=3, stride=1, padding=1)
        is_peak_mask = (density == max_pooled) & (density > 0.3)
        
        for b in range(B):
            seeds = grid[is_peak_mask[b, 0].view(-1)]
            probs = density[b, 0][is_peak_mask[b, 0]]
            
            sorted_idx = torch.argsort(probs, descending=True)
            seeds, probs = seeds[sorted_idx], probs[sorted_idx]
            
            keep_mask = torch.ones(seeds.shape[0], dtype=torch.bool, device=device)
            for idx in range(seeds.shape[0]):
                if not keep_mask[idx]:
                    continue
                clash_mask = torch.sum((seeds[idx+1:] - seeds[idx]) ** 2, dim=-1).sqrt() < 0.6
                keep_mask[idx+1:][clash_mask] = False
                
            seeds, probs = seeds[keep_mask][:M], probs[keep_mask][:M]
            num_to_copy = seeds.shape[0]
            if num_to_copy > 0:
                out_coords[b, :num_to_copy] = seeds
                out_values[b, :num_to_copy] = probs
                out_mask[b, :num_to_copy] = True
                
        return out_coords, out_values, out_mask


# ==============================================================================
# DYNAMIC SIMULATOR & TRAINING LOGIC
# ==============================================================================

def crop_and_rasterize_dynamic(structures: list, return_coords: bool = False, is_training: bool = False) -> tuple:
    coords, res_indices = random.choice(structures)
    center = coords[torch.randint(0, coords.shape[0], (1,)).item()]
    
    mask = torch.all((coords >= center - 4.0) & (coords <= center + 4.0), dim=-1)
    cropped_coords = coords[mask] - center + 4.0
    cropped_res = res_indices[mask]
    
    sigma = random.uniform(0.8, 1.8) if is_training else 1.2
    noise = random.uniform(0.01, 0.08) if is_training else 0.04
    
    inp_density = F.relu(coords_to_density(cropped_coords, sigma) + torch.randn(32, 32, 32, device=coords.device) * noise)
    target_atom = coords_to_binary_grid(cropped_coords)
    target_res = coords_to_residue_grid(cropped_coords, cropped_res)
    
    if return_coords:
        return inp_density, target_atom, target_res, cropped_coords, cropped_res
    return inp_density, target_atom, target_res


if __name__ == "__main__":
    torch.manual_seed(RANDOM_SEED)
    random.seed(RANDOM_SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

    # Download dataset
    train_files = [download_pdb_cif(pid) for pid in TRAIN_PDB_IDS[:16]]  # Load all 16 training structures
    test_file = download_pdb_cif(TEST_PDB_ID)
    
    train_structures = []
    for filepath in train_files:
        coords, indices = load_coords_biotite(filepath)
        train_structures.append((coords.to(device), indices.to(device)))
        print(f"Loaded {os.path.basename(filepath)} | Atoms: {len(coords)}")
        
    val_dataset = [crop_and_rasterize_dynamic(train_structures, is_training=True) for _ in range(20)]

    # Model Setup
    atom_model = UNet3D(1, 1).to(device)
    residue_model = UNet3D(1, len(RESIDUE_MAP) + 1).to(device)
    
    optimizer = torch.optim.Adam(itertools.chain(atom_model.parameters(), residue_model.parameters()), lr=0.001)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=60, eta_min=1e-5)
    criterion_atom = BCEDiceLoss()
    
    # Standard CrossEntropyLoss to enforce a strong spatial localization of the residue classes
    criterion_residue = nn.CrossEntropyLoss()
    
    peak_finder = BatchedMeanShiftPeakFinder3D().to(device)

    # HIGH-ACCURACY SCALED-UP PRODUCTION TRAINING CONFIGURATION
    epochs_to_run = 35        # Run for 35 epochs
    crops_per_epoch = 1500    # Use 1500 crops per epoch
    steps_per_epoch = crops_per_epoch // 8

    print(f"\nTraining models ({epochs_to_run} epochs with {crops_per_epoch} crops per epoch)...")
    for epoch in range(1, epochs_to_run + 1):
        atom_model.train(); residue_model.train()
        train_loss = 0.0
        
        for step in range(steps_per_epoch):
            batch = [crop_and_rasterize_dynamic(train_structures, is_training=True) for _ in range(8)]
            x = torch.stack([b[0] for b in batch]).unsqueeze(1).to(device)
            y_a = torch.stack([b[1] for b in batch]).unsqueeze(1).to(device)
            y_r = torch.stack([b[2] for b in batch]).long().to(device)
            
            x, y_a, y_r = augment_batch_3d_joint(x, y_a, y_r)
            
            optimizer.zero_grad()
            pred_a, ds_a = atom_model(x, return_ds=True)
            pred_r, ds_r = residue_model(x, return_ds=True)
            
            loss_a = criterion_atom(torch.sigmoid(pred_a), y_a) + 0.5 * criterion_atom(torch.sigmoid(ds_a), F.max_pool3d(y_a, 2, 2))
            loss_r = criterion_residue(pred_r, y_r) + 0.5 * criterion_residue(ds_r, F.max_pool3d(y_r.float().unsqueeze(1), 2, 2).squeeze(1).long())
            
            loss = loss_a + loss_r
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
        # Validation
        atom_model.eval(); residue_model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for v_x, v_ya, v_yr in val_dataset:
                vx_tensor, vy_a_tensor, vy_r_tensor = v_x.unsqueeze(0).unsqueeze(0).to(device), v_ya.unsqueeze(0).unsqueeze(0).to(device), v_yr.unsqueeze(0).to(device)
                p_a, ds_a = atom_model(vx_tensor, return_ds=True)
                p_r, ds_r = residue_model(vx_tensor, return_ds=True)
                
                loss_a = criterion_atom(torch.sigmoid(p_a), vy_a_tensor) + 0.5 * criterion_atom(torch.sigmoid(ds_a), F.max_pool3d(vy_a_tensor, 2, 2))
                loss_r = criterion_residue(p_r, vy_r_tensor) + 0.5 * criterion_residue(ds_r, F.max_pool3d(vy_r_tensor.float().unsqueeze(1), 2, 2).squeeze(1).long())
                val_loss += (loss_a + loss_r).item()
                
        scheduler.step()
        print(f"Epoch {epoch:02d}/{epochs_to_run:02d} | Train Loss: {train_loss/steps_per_epoch:.3f} | Val Loss: {val_loss/len(val_dataset):.3f}")

    # Evaluation on Unseen Crambin
    print("\nEvaluating on unseen Crambin (1CRN)...\n")
    test_coords, test_indices = load_coords_biotite(test_file)
    test_dataset = [crop_and_rasterize_dynamic([(test_coords.to(device), test_indices.to(device))], return_coords=True) for _ in range(5)]
    
    total_gt, total_matched, total_correct = 0, 0, 0
    with torch.no_grad():
        for t_in, t_tgt_a, t_tgt_r, t_coords, t_res_indices in test_dataset:
            t_in_batch = t_in.unsqueeze(0).unsqueeze(0).to(device)
            pred_density = F.relu(torch.sigmoid(atom_model(t_in_batch)))
            pred_coords, _, pred_mask = peak_finder(pred_density)
            pred_res_logits = residue_model(t_in_batch)
            
            p_coords, p_mask = pred_coords[0].cpu(), pred_mask[0].cpu()
            n_peaks = p_mask.sum().item()
            total_gt += len(t_coords)
            
            for i, gt_c in enumerate(t_coords):
                gt_idx = t_res_indices[i].item()
                if n_peaks > 0:
                    dists = torch.norm(p_coords[:n_peaks] - gt_c.cpu(), dim=-1)
                    min_dist, min_idx = dists.min(dim=0)
                    if min_dist.item() <= 1.0:
                        total_matched += 1
                        grid_idx = torch.clamp(torch.round(p_coords[min_idx] / (8.0/31)).long(), 0, 31)
                        logits = pred_res_logits[0, :, grid_idx[0], grid_idx[1], grid_idx[2]]
                        pred_class = torch.argmax(logits[1:]).item() + 1
                        if pred_class == gt_idx:
                            total_correct += 1
                        if total_matched <= 5:
                            print(f"      [Debug Match {total_matched}] GT Class: {gt_idx} | Pred Class: {pred_class} | Logits sample: {logits[:5].tolist()}")

    rec = (total_matched / total_gt) * 100 if total_gt > 0 else 0.0
    cls = (total_correct / total_matched) * 100 if total_matched > 0 else 0.0
    print(f"\nOverall Recovery: {rec:.1f}% ({total_matched}/{total_gt} atoms resolved within 1.0 Å)")
    print(f"Overall Classification: {cls:.1f}% ({total_correct}/{total_matched} correct residue types)")